
<div class="alert alert-block alert-success">
<b> Assign : </b> HyperParameter Optimization.

In [26]:
!pip install optuna


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 0 - Import the Necessary Libraries

In [27]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

---
## Step 1: Load the Dataset

#### Overview
The Fashion-MNIST dataset is loaded natively through PyTorch's `torchvision.datasets` module, providing a lightweight benchmark dataset for image classification tasks. This approach eliminates the need for manual file downloads, folder management, or complex data pipelines, significantly reducing storage requirements and system memory overhead.

#### Dataset Characteristics
- **Dataset Type**: Fashion-MNIST (clothing item classification)
- **Image Dimensions**: 28x28 pixels (grayscale)
- **Number of Classes**: 10 (T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot)
- **Training Samples**: 60,000 images
- **Test Samples**: 10,000 images
- **Total Size**: ~30 MB (lightweight and memory-efficient)

In [28]:
# 2. Define Data Transformations

transform = transforms.Compose([
    transforms.ToTensor(),  # Converts PIL image to tensor (0-255 → 0-1)
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1] range
])

# 3. Load Training Dataset

train_dataset = datasets.FashionMNIST(
    root='./data',          # Local storage directory
    train=True,             # Load training split
    download=True,          # Auto-download if not present
    transform=transform     # Apply transformations
)
full_dataset = train_dataset

# 4. Load Test Dataset

test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,            # Load test split
    download=True,
    transform=transform
)

In [29]:
print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")

Training samples: 60000
Test samples: 10000
Image shape: torch.Size([1, 28, 28])


---
## Step 2: Split the Data

#### Overview
The loaded dataset is strategically partitioned into three distinct subsets: Training, Validation, and Testing. This three-way split is essential for unbiased model evaluation and effective hyperparameter optimization. Each partition serves a specific purpose in the machine learning workflow, ensuring the model's performance is properly assessed on unseen data.

#### Split Ratios and Purposes

| Partition | Percentage | Purpose | Usage |
|-----------|------------|---------|-------|
| **Training** | 80% | Model weight updates | Used during forward/backward propagation to learn patterns |
| **Validation** | 10% | Hyperparameter evaluation | Used exclusively inside the HPO loop to grade each configuration |
| **Testing** | 10% | Final model evaluation | Held out completely until the very end for unbiased assessment |

#### Implementation Actions

**1. Define Split Sizes**
```python
from torch.utils.data import random_split

# Calculate partition sizes
dataset_size = len(full_dataset)
train_size = int(0.8 * dataset_size)   # 80% for training
val_size = int(0.1 * dataset_size)     # 10% for validation
# Remaining 10% will be automatically assigned to test

In [30]:
# **1. Define Split Sizes**
# ```python

from torch.utils.data import random_split

# Calculate partition sizes
dataset_size = len(full_dataset)
train_size = int(0.8 * dataset_size)   # 80% for training
val_size = int(0.1 * dataset_size)     # 10% for validation
# Remaining 10% will be automatically assigned to test
test_size = dataset_size - train_size - val_size  # Ensures no leftover samples

In [31]:
## 2. Perform Random Split

# Split the dataset with fixed seed for reproducibility
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, 
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)
## 3. Verify Split Distribution

# Calculate actual sizes
print('Dataset Statistics : ')
print(f"Full datset : {len(full_dataset) } samples ({len(full_dataset)/dataset_size*100 :.1f}%)")
print(f"Training split: {len(train_dataset)} samples ({len(train_dataset)/dataset_size*100:.1f}%)")
print(f"Validation split: {len(val_dataset)} samples ({len(val_dataset)/dataset_size*100:.1f}%)")
print(f"Test split: {len(test_dataset)} samples ({len(test_dataset)/dataset_size*100:.1f}%)")

Dataset Statistics : 
Full datset : 60000 samples (100.0%)
Training split: 48000 samples (80.0%)
Validation split: 6000 samples (10.0%)
Test split: 6000 samples (10.0%)


---
## Step 3: Perform Preprocessing

#### Overview
The raw image data undergoes essential preprocessing transformations to convert it into mathematically stable tensors optimized for CPU training. This step involves reshaping image dimensions, normalizing pixel values to a consistent range, and creating efficient data loaders that stream data in batches to prevent memory bottlenecks during training.

#### Preprocessing Pipeline

The preprocessing pipeline consists of three main operations performed sequentially:

#### 1. Tensor Reshaping

**Purpose**: Convert raw image data into the standard tensor format expected by deep learning frameworks.

**Implementation**:
```python
# PyTorch format: [Batch, Channels, Height, Width]
# Original image: 28x28 grayscale → Tensor shape: [1, 28, 28]
# After DataLoader batching: [64, 1, 28, 28]

# TensorFlow format: [Batch, Height, Width, Channels]
# Original image: 28x28 grayscale → Tensor shape: [28, 28, 1]
# After DataLoader batching: [64, 28, 28, 1]

In [44]:
## 3. Data Loader Creation
from torch.utils.data import DataLoader

# Set fixed batch size to optimize memory usage
BATCH_SIZE = 64

# Create data loaders for each split
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE,
    shuffle=True,           # Randomize order for training
    num_workers=2,          # Parallel data loading
    pin_memory=True         # Speed up data transfer to CPU/GPU
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,          # No shuffling for validation
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,          # No shuffling for testing
    num_workers=2,
    pin_memory=True
)
# Verify preprocessing
sample_batch = next(iter(train_loader))
images, labels = sample_batch
print(f"Batch shape: {images.shape}")  # Expected: [64, 1, 28, 28]
print(f"Label shape: {labels.shape}")   # Expected: [64]
print(f"Pixel range: [{images.min():.2f}, {images.max():.2f}]")  # Expected: [-1.0, 1.0]

Batch shape: torch.Size([64, 1, 28, 28])
Label shape: torch.Size([64])
Pixel range: [-1.00, 1.00]


In [33]:
# Check data loader iteration
for batch_idx, (data, targets) in enumerate(train_loader):
    print(f"Batch {batch_idx}: Data shape {data.shape}, Targets shape {targets.shape}")
    if batch_idx == 2:  # Check only first few batches
        break

Batch 0: Data shape torch.Size([64, 1, 28, 28]), Targets shape torch.Size([64])
Batch 1: Data shape torch.Size([64, 1, 28, 28]), Targets shape torch.Size([64])
Batch 2: Data shape torch.Size([64, 1, 28, 28]), Targets shape torch.Size([64])


---
## Step 4: Build a Parametric Model Template

#### Overview
A flexible and dynamic model construction function is created that builds a convolutional neural network based on structural arguments. Instead of using a rigid, hard-coded architecture, this parametric approach allows the model to accept dynamic parameters like number of filters and learning rate, enabling efficient hyperparameter optimization. The architecture follows a simple but effective pattern: Convolutional Layer → Max Pooling → Flatten → Dense Output Layer.

#### Model Architecture

The parametric model template consists of four sequential layers:

- Input Image [Batch, 1, 28, 28]
- ###            ↓
- Conv2D Layer (Dynamic Filters)
- ###            ↓
- Max Pooling Layer (2x2)
- ###            ↓
- Flatten Layer
- ###            ↓
- Dense Output Layer (10 units)
- ###            ↓
- Output [Batch, 10]

In [34]:
## **1. Define the Parametric Model Class**

import torch
import torch.nn as nn
import torch.nn.functional as F

class FlexibleCNN(nn.Module):
    """
    A flexible CNN model that accepts dynamic architectural parameters.
    
    Args:
        num_filters (int): Number of filters in the convolutional layer
        learning_rate (float): Learning rate for the optimizer (not used in model init)
        num_classes (int): Number of output classes (default: 10)
    """
    
    def __init__(self, num_filters=32, num_classes=10):
        super(FlexibleCNN, self).__init__()
        
        # Layer 1: Convolutional 2D Layer
        # Input: [Batch, 1, 28, 28] → Output: [Batch, num_filters, 26, 26]
        self.conv1 = nn.Conv2d(
            in_channels=1,           # Grayscale input
            out_channels=num_filters, # Dynamic number of filters
            kernel_size=3,           # 3x3 convolution kernel
            stride=1,                # Step size
            padding=1                # Keep spatial dimensions same
        )
        
        # Layer 2: Max Pooling Layer
        # Input: [Batch, num_filters, 28, 28] → Output: [Batch, num_filters, 14, 14]
        self.pool = nn.MaxPool2d(
            kernel_size=2,           # 2x2 pooling window
            stride=2                 # Stride of 2 (no overlap)
        )
        
        # Layer 3: Fully Connected Output Layer
        # Input: [Batch, num_filters * 14 * 14] → Output: [Batch, 10]
        self.fc1 = nn.Linear(
            in_features=num_filters * 14 * 14,  # Dynamic input size based on filters
            out_features=num_classes            # 10 Fashion-MNIST classes
        )
        
    def forward(self, x):
        """
        Forward pass through the network.
        
        Args:
            x (torch.Tensor): Input tensor of shape [Batch, 1, 28, 28]
            
        Returns:
            torch.Tensor: Output logits of shape [Batch, 10]
        """
        # Apply convolution + ReLU activation
        x = F.relu(self.conv1(x))        # [Batch, num_filters, 28, 28]
        
        # Apply max pooling
        x = self.pool(x)                 # [Batch, num_filters, 14, 14]
        
        # Flatten the spatial dimensions
        x = x.view(x.size(0), -1)        # [Batch, num_filters * 14 * 14]
        
        # Apply fully connected layer (no activation - logits for CrossEntropy)
        x = self.fc1(x)                  # [Batch, 10]
        
        return x
    def build_model(num_filters=32, num_classes=10):
        """
        Build a sequential CNN model (alternative to class-based approach).
    
        Args:
            num_filters (int): Number of filters in conv layer
            num_classes (int): Number of output classes
        
        Returns:
            nn.Sequential: Sequential model with the defined architecture
        """
        model = nn.Sequential(
            # Conv2D Layer
            nn.Conv2d(1, num_filters, kernel_size=3, padding=1),
            nn.ReLU(),
        
            # Max Pooling Layer
            nn.MaxPool2d(kernel_size=2, stride=2),
        
            # Flatten Layer
            nn.Flatten(),
        
            # Dense Output Layer
            nn.Linear(num_filters * 14 * 14, num_classes)
        )
    
        return model

In [35]:
## 4. Output Layer Configuration
# The output layer MUST have exactly 10 units
NUM_CLASSES = 10  # Matches Fashion-MNIST classes

# Cross-Entropy Loss is automatically applied
criterion = nn.CrossEntropyLoss()  # Combines LogSoftmax + NLLLoss

# The output from the model (logits) is compatible with CrossEntropyLoss
# No softmax needed in the model definition

In [36]:
def create_model(num_filters=32, learning_rate=0.001, optimizer_name='Adam'):
    """
    Complete model setup with optimizer and criterion.
    
    Args:
        num_filters (int): Number of filters in conv layer
        learning_rate (float): Learning rate for optimizer
        optimizer_name (str): 'Adam' or 'SGD'
        
    Returns:
        tuple: (model, optimizer, criterion)
    """
    # Instantiate the model with dynamic filters
    model = FlexibleCNN(num_filters=num_filters, num_classes=10)
    
    # Define loss function
    criterion = nn.CrossEntropyLoss()
    
    # Select optimizer based on hyperparameter
    if optimizer_name.lower() == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    elif optimizer_name.lower() == 'sgd':
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")
    
    return model, optimizer, criterion

In [37]:
def print_model_summary(model, input_size=(1, 28, 28)):
    """
    Print detailed model architecture with parameter counts.
    """
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print("=" * 60)
    print("MODEL ARCHITECTURE SUMMARY")
    print("=" * 60)
    print(model)
    print("-" * 60)
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print(f"Input Shape: {input_size}")
    print(f"Output Shape: (Batch, 10)")
    print("=" * 60)

# Example usage
model = FlexibleCNN(num_filters=32)
print_model_summary(model)

MODEL ARCHITECTURE SUMMARY
FlexibleCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=6272, out_features=10, bias=True)
)
------------------------------------------------------------
Total Parameters: 63,050
Trainable Parameters: 63,050
Input Shape: (1, 28, 28)
Output Shape: (Batch, 10)


--- 
## Step 5: Setup the HPO Framework (Optuna Integration)

#### Overview
The Hyperparameter Optimization (HPO) framework is established by wrapping the entire model building, training, and evaluation pipeline inside an Optuna objective function. This integration enables systematic exploration of hyperparameter combinations, automatic pruning of underperforming trials, and efficient search for the optimal configuration. Optuna's built-in features like trial suggestions, pruning, and study management streamline the optimization process.

#### Optuna Workflow Architecture
```text
Optuna Study
↓
Objective Function (Called for each trial)
↓
├── Suggest Hyperparameters
│   ├── learning_rate (log-uniform: 1e-4 to 1e-2)
│   ├── optimizer (categorical: Adam/SGD)
│   └── num_filters (integer: 16, 32, 48, 64)
↓
├── Build Model with Suggested Parameters
↓
├── Train Model for 2 Epochs
│   └── After each epoch → Report validation accuracy
↓
├── Pruning Check
│   └── If underperforming → trial.should_prune() → Kill trial
↓
└── Return Final Validation Accuracy


In [38]:
# **1. Define the Objective Function**

import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

def objective(trial):
    """
    Optuna objective function that defines the HPO space and training pipeline.
    
    Args:
        trial (optuna.trial.Trial): Optuna trial object for parameter suggestions
        
    Returns:
        float: Validation accuracy for the trial
    """
    
    # ============================================
    # STEP 1: Suggest Hyperparameters
    # ============================================
    
    # Learning Rate: Log-uniform distribution for better exploration
    lr = trial.suggest_float(
        "lr", 
        1e-4,           # Lower bound (0.0001)
        1e-2,           # Upper bound (0.01)
        log=True        # Logarithmic scale for better sampling
    )
    
    # Optimizer: Categorical choice between Adam and SGD
    optimizer_name = trial.suggest_categorical(
        "optimizer", 
        ["Adam", "SGD"]
    )
    
    # Number of Filters: Integer choices with 16-step increments
    num_filters = trial.suggest_int(
        "filters", 
        16,             # Minimum
        64,             # Maximum
        step=16         # Step size (16, 32, 48, 64)
    )
    
    # ============================================
    # STEP 2: Build Model with Suggested Parameters
    # ============================================
    
    # Create the parametric model with dynamic filters
    model = FlexibleCNN(num_filters=num_filters, num_classes=10)
    
    # Select optimizer based on suggestion
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:  # SGD
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    
    # Define loss function
    criterion = nn.CrossEntropyLoss()
    
    # ============================================
    # STEP 3: Training Loop (2 Epochs)
    # ============================================
    
    # Set device (CPU for this implementation)
    device = torch.device("cpu")
    model.to(device)
    
    # Training parameters
    EPOCHS = 2
    BATCH_SIZE = 64
    
    # Get data loaders (global variables from previous steps)
    train_loader = get_train_loader()  # Assuming defined globally
    val_loader = get_val_loader()      # Assuming defined globally
    
    # Training loop
    for epoch in range(EPOCHS):
        # --- Training Phase ---
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (data, targets) in enumerate(train_loader):
            # Move data to device
            data, targets = data.to(device), targets.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, targets)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Track metrics
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += targets.size(0)
            train_correct += predicted.eq(targets).sum().item()
        
        train_accuracy = 100.0 * train_correct / train_total
        
        # --- Validation Phase ---
        model.eval()
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for data, targets in val_loader:
                data, targets = data.to(device), targets.to(device)
                outputs = model(data)
                _, predicted = outputs.max(1)
                val_total += targets.size(0)
                val_correct += predicted.eq(targets).sum().item()
        
        val_accuracy = 100.0 * val_correct / val_total
        
        # ============================================
        # STEP 4: Pruning Anchor
        # ============================================
        
        # Report intermediate objective value to Optuna
        trial.report(val_accuracy, epoch)
        
        # Handle pruning based on the intermediate value
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
        # Optional: Print progress
        print(f"Trial {trial.number}, Epoch {epoch+1}/{EPOCHS}: "
              f"Train Acc: {train_accuracy:.2f}%, Val Acc: {val_accuracy:.2f}%")
    
    # ============================================
    # STEP 5: Return Final Validation Accuracy
    # ============================================
    
    return val_accuracy

In [39]:
# Global data loaders (created in Step 3)
train_loader = None
val_loader = None
test_loader = None

def set_data_loaders(train, val, test):
    """Set global data loaders for use in objective function."""
    global train_loader, val_loader, test_loader
    train_loader = train
    val_loader = val
    test_loader = test

def objective_with_data(trial, train_loader, val_loader):
    """
    Objective function with explicit data loader parameters.
    """
    # Hyperparameter suggestions
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    num_filters = trial.suggest_int("filters", 16, 64, step=16)
    
    # Build model
    model = FlexibleCNN(num_filters=num_filters, num_classes=10)
    optimizer = get_optimizer(model, optimizer_name, lr)
    criterion = nn.CrossEntropyLoss()
    
    device = torch.device("cpu")
    model.to(device)
    
    # Training loop with pruning
    for epoch in range(2):
        # Train for one epoch
        train_accuracy = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_accuracy = validate_one_epoch(model, val_loader, criterion, device)
        
        # Report and prune
        trial.report(val_accuracy, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    
    return val_accuracy

def train_one_epoch(model, loader, optimizer, criterion, device):
    """Helper function to train for one epoch."""
    model.train()
    correct = 0
    total = 0
    
    for data, targets in loader:
        data, targets = data.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    return 100.0 * correct / total

def validate_one_epoch(model, loader, criterion, device):
    """Helper function to validate for one epoch."""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return 100.0 * correct / total

def get_optimizer(model, name, lr):
    """Helper function to get optimizer."""
    if name == "Adam":
        return torch.optim.Adam(model.parameters(), lr=lr)
    else:  # SGD
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)

In [40]:
def objective_advanced(trial):
    """
    Advanced objective function with multiple reporting intervals.
    """
    # Hyperparameter suggestions
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    num_filters = trial.suggest_int("filters", 16, 64, step=16)
    
    # Build model and optimizer
    model = FlexibleCNN(num_filters=num_filters, num_classes=10)
    optimizer = get_optimizer(model, optimizer_name, lr)
    criterion = nn.CrossEntropyLoss()
    device = torch.device("cpu")
    model.to(device)
    
    # Track best validation accuracy
    best_val_accuracy = 0.0
    
    # Training for 2 epochs
    for epoch in range(2):
        # Train for one epoch
        model.train()
        for batch_idx, (data, targets) in enumerate(train_loader):
            data, targets = data.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            # Optional: Report on batch level for more granular pruning
            if batch_idx % 100 == 0:  # Report every 100 batches
                # Quick validation on subset
                val_acc = quick_validate(model, val_loader, device)
                intermediate_value = epoch + batch_idx / len(train_loader)
                trial.report(val_acc, intermediate_value)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
        
        # Full validation at end of epoch
        val_accuracy = full_validate(model, val_loader, device)
        best_val_accuracy = max(best_val_accuracy, val_accuracy)
        
        # Report and prune based on full epoch validation
        trial.report(val_accuracy, epoch + 1)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    
    return best_val_accuracy

In [41]:
def objective_robust(trial):
    """
    Robust objective function with error handling and logging.
    """
    try:
        # Suggest hyperparameters
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
        num_filters = trial.suggest_int("filters", 16, 64, step=16)
        
        # Log hyperparameters
        print(f"\nTrial {trial.number}:")
        print(f"  lr: {lr:.6f}")
        print(f"  optimizer: {optimizer_name}")
        print(f"  filters: {num_filters}")
        
        # Build model
        model = FlexibleCNN(num_filters=num_filters, num_classes=10)
        optimizer = get_optimizer(model, optimizer_name, lr)
        criterion = nn.CrossEntropyLoss()
        device = torch.device("cpu")
        model.to(device)
        
        best_val_accuracy = 0.0
        
        # Training loop with pruning
        for epoch in range(2):
            # Train
            model.train()
            running_loss = 0.0
            for data, targets in train_loader:
                data, targets = data.to(device), targets.to(device)
                optimizer.zero_grad()
                outputs = model(data)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
            
            # Validate
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for data, targets in val_loader:
                    data, targets = data.to(device), targets.to(device)
                    outputs = model(data)
                    _, predicted = outputs.max(1)
                    total += targets.size(0)
                    correct += predicted.eq(targets).sum().item()
            
            val_accuracy = 100.0 * correct / total
            best_val_accuracy = max(best_val_accuracy, val_accuracy)
            
            # Report and prune
            trial.report(val_accuracy, epoch)
            if trial.should_prune():
                print(f"  → Trial pruned at epoch {epoch+1} (accuracy: {val_accuracy:.2f}%)")
                raise optuna.exceptions.TrialPruned()
            
            print(f"  Epoch {epoch+1}: Val Acc = {val_accuracy:.2f}%")
        
        print(f"  → Best Val Acc: {best_val_accuracy:.2f}%")
        return best_val_accuracy
        
    except Exception as e:
        print(f"  ✗ Trial failed with error: {e}")
        raise e

---
## Step 6: Execute the Optimization Study

#### Overview
The Optuna optimization study is initialized and executed to systematically search for the optimal hyperparameter configuration. The study orchestrates multiple trials, each evaluating a unique combination of hyperparameters through the objective function. With the direction set to maximize validation accuracy, the study explores the hyperparameter space using efficient sampling strategies, automatically tracking the best-performing configurations across all trials.

#### Study Initialization



In [45]:
# ============================================
# STEP 6: EXECUTE OPTIMIZATION (DIRECT PASSING)
# ============================================

import optuna
from optuna.trial import TrialState
import warnings
import time
from datetime import datetime
import numpy as np
warnings.filterwarnings('ignore')

def objective_with_loaders(trial, train_loader_local, val_loader_local):
    """
    Objective function that accepts data loaders as parameters.
    """
    # Suggest hyperparameters
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    num_filters = trial.suggest_int("filters", 16, 64, step=16)
    
    # Build model
    model = FlexibleCNN(num_filters=num_filters, num_classes=10)
    
    # Select optimizer
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:  # SGD
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    
    criterion = nn.CrossEntropyLoss()
    device = torch.device("cpu")
    model.to(device)
    
    best_val_accuracy = 0.0
    
    # Training loop - 2 epochs
    for epoch in range(2):
        # --- Training Phase ---
        model.train()
        train_correct = 0
        train_total = 0
        
        for data, targets in train_loader_local:
            data, targets = data.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            _, predicted = outputs.max(1)
            train_total += targets.size(0)
            train_correct += predicted.eq(targets).sum().item()
        
        train_accuracy = 100.0 * train_correct / train_total
        
        # --- Validation Phase ---
        model.eval()
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for data, targets in val_loader_local:
                data, targets = data.to(device), targets.to(device)
                outputs = model(data)
                _, predicted = outputs.max(1)
                val_total += targets.size(0)
                val_correct += predicted.eq(targets).sum().item()
        
        val_accuracy = 100.0 * val_correct / val_total
        best_val_accuracy = max(best_val_accuracy, val_accuracy)
        
        # Report and prune
        trial.report(val_accuracy, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
        print(f"Trial {trial.number}, Epoch {epoch+1}/2: "
              f"Train Acc: {train_accuracy:.2f}%, Val Acc: {val_accuracy:.2f}%")
    
    return best_val_accuracy

# ============================================
# CHECK DATA LOADERS BEFORE RUNNING
# ============================================

print("="*70)
print("DATA LOADER VERIFICATION")
print("="*70)

# Check if data loaders exist
if 'train_loader' not in globals() or train_loader is None:
    print("❌ ERROR: train_loader is not initialized!")
    print("Please run Step 3 first to create the data loaders.")
    raise ValueError("train_loader is None. Run Step 3 first!")

if 'val_loader' not in globals() or val_loader is None:
    print("❌ ERROR: val_loader is not initialized!")
    print("Please run Step 3 first to create the data loaders.")
    raise ValueError("val_loader is None. Run Step 3 first!")

print(f"✅ Data loaders verified:")
print(f"  Train loader: {len(train_loader)} batches")
print(f"  Validation loader: {len(val_loader)} batches")
print(f"  Test loader: {len(test_loader)} batches")

# ============================================
# EXECUTE OPTIMIZATION
# ============================================

print("\n" + "="*70)
print("FASHION-MNIST HYPERPARAMETER OPTIMIZATION")
print("="*70)
print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("-"*70)

# Create study
study = optuna.create_study(
    direction="maximize",
    study_name="fashion_mnist_hpo_fixed_v2",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=2,
        n_warmup_steps=0,
        interval_steps=1
    )
)

print("✓ Study created successfully")
print("-"*70)

# Start optimization - PASS LOADERS DIRECTLY
print("Starting optimization with 10 trials...")
print("-"*70)

start_time = time.time()

study.optimize(
    lambda trial: objective_with_loaders(trial, train_loader, val_loader),
    n_trials=10,
    timeout=1200,
    n_jobs=1,
    catch=(Exception,)
)

elapsed_time = time.time() - start_time

# ============================================
# RESULTS
# ============================================

print("\n" + "="*70)
print("OPTIMIZATION COMPLETED")
print("="*70)
print(f"Total Time: {elapsed_time/60:.1f} minutes")
print(f"Total Trials: {len(study.trials)}")
print("-"*70)

# Display best results
if study.best_value is not None:
    print("\n🏆 BEST CONFIGURATION")
    print("-"*70)
    print(f"Validation Accuracy: {study.best_value:.2f}%")
    print(f"Hyperparameters:")
    for key, value in study.best_params.items():
        print(f"  → {key}: {value}")
    
    # Summary statistics
    completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]
    if completed_trials:
        accuracies = [t.value for t in completed_trials if t.value is not None]
        if accuracies:
            print("\n📊 STATISTICS")
            print("-"*70)
            print(f"Mean Accuracy: {sum(accuracies)/len(accuracies):.2f}%")
            print(f"Std Dev: {np.std(accuracies):.2f}%")
            print(f"Min Accuracy: {min(accuracies):.2f}%")
            print(f"Max Accuracy: {max(accuracies):.2f}%")

# Trial summary
print("\n📋 TRIAL SUMMARY")
print("-"*70)
for i, trial in enumerate(study.trials):
    status = trial.state.name
    value = f"{trial.value:.2f}%" if trial.value is not None else "N/A"
    print(f"Trial {i}: {status} - Acc: {value} - Params: {trial.params}")

print("="*70)

[I 2026-07-31 12:19:55,725] A new study created in memory with name: fashion_mnist_hpo_fixed_v2


DATA LOADER VERIFICATION
✅ Data loaders verified:
  Train loader: 750 batches
  Validation loader: 94 batches
  Test loader: 94 batches

FASHION-MNIST HYPERPARAMETER OPTIMIZATION
Start Time: 2026-07-31 12:19:55
----------------------------------------------------------------------
✓ Study created successfully
----------------------------------------------------------------------
Starting optimization with 10 trials...
----------------------------------------------------------------------
Trial 0, Epoch 1/2: Train Acc: 83.46%, Val Acc: 87.92%


[I 2026-07-31 12:20:55,220] Trial 0 finished with value: 88.8 and parameters: {'lr': 0.0005611516415334506, 'optimizer': 'Adam', 'filters': 48}. Best is trial 0 with value: 88.8.


Trial 0, Epoch 2/2: Train Acc: 88.47%, Val Acc: 88.80%
Trial 1, Epoch 1/2: Train Acc: 82.69%, Val Acc: 87.32%


[I 2026-07-31 12:22:03,822] Trial 1 finished with value: 88.25 and parameters: {'lr': 0.0002051338263087451, 'optimizer': 'Adam', 'filters': 64}. Best is trial 0 with value: 88.8.


Trial 1, Epoch 2/2: Train Acc: 87.77%, Val Acc: 88.25%
Trial 2, Epoch 1/2: Train Acc: 85.23%, Val Acc: 88.90%


[W 2026-07-31 12:22:55,875] Trial 2 failed with parameters: {'lr': 0.0015930522616241021, 'optimizer': 'Adam', 'filters': 64} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "e:\Z_Arise\rika_dl_env\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Soclab2-13\AppData\Local\Temp\ipykernel_5804\1628726917.py", line 141, in <lambda>
    lambda trial: objective_with_loaders(trial, train_loader, val_loader),
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Soclab2-13\AppData\Local\Temp\ipykernel_5804\1628726917.py", line 50, in objective_with_loaders
    loss.backward()
  File "e:\Z_Arise\rika_dl_env\Lib\site-packages\torch\_tensor.py", line 623, in backward
    torch.autograd.backward(
  File "e:\Z_Arise\rika_dl_env\Lib\site-packages\torch\autograd\__init__.py", line 395, in backward
    _engine_run_

KeyboardInterrupt: 